# Phase 2 - E7 ensemble + document-level post-processing
Average the tag probabilities of the three A d768 runs (E2 LoRMA, E1 full FT bidirectional, E1 full FT causal; same tokenizer), decode once (BIO Viterbi + word snapping), then:
- **party roles**: APP/RESP set by the side of 'Versus' in the case title, later mentions follow the header names
- **name propagation**: every standalone repeat of a predicted person/party name gets the same label (in the training gold every repeat of an annotated string is annotated, 0 exceptions)

Code: `Phase2/src/ensemble.py`, `Phase2/src/postprocess.py`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
from ensemble import ensemble_predict, ensemble_submission
from ner_data import load_split
from evaluate import strict_scores, format_scores
from postprocess import resolve_party_roles, propagate_strings, NAME_LABELS

R = "C:/kaggle/working/cs781/phase2/runs/"
MEMBERS = [R + "E2_A_lorma/A_d768_lorma_684310", R + "E1_A_full/A_d768_full_8fcd22", R + "E1_A_full/A_d768_full_5c4900"]
texts, gold, test, split = load_split()
g = {d: gold[d] for d in split["val"]}

## Validation

In [ ]:
p = ensemble_predict(MEMBERS, "val")
roles = {d: resolve_party_roles(texts[d], p[d]) for d in p}
names = {d: propagate_strings(texts[d], roles[d], labels=NAME_LABELS) for d in p}
for tag, pp in [("ensemble", p), ("+ party roles", roles), ("+ party roles + name propagation", names)]:
    sc = strict_scores(pp, g); print(f"{tag:36s} macro {sc['macro_f1']:.4f}  micro {sc['micro_f1']:.4f}")
print(format_scores(strict_scores(names, g)))

## Test submissions (already written; re-running creates new timestamped folders)

In [ ]:
# ensemble_submission(MEMBERS, "ens3_roles_propnames", roles=True, propagate_names=True)
# ensemble_submission(MEMBERS, "ens3_roles", roles=True, propagate_names=False)